# AI-Based Predictive Maintenance for Mining Equipment
**Group 9 · Saskatchewan Polytechnic · PROJ 613**

**Upload one file → recognize the input and its source format → resolve the equipment mapping → load its saved fault model → display the condition and probability.**

In VS Code, select the project’s `.venv` kernel and run all cells. If needed, install the upload widget in that kernel with `%pip install "ipywidgets>=8,<9"`, then restart the kernel. Use a complete, supported sensor recording or a supported fault-data image. Leave the mapping on **Automatic** for ordinary use. If you know the source of a shared-format file, choose its verified laboratory mapping before predicting.

Sensor CSVs support Ottawa, UODS, MCC5-THU, JUST and named CORA vibration recordings. See the README for the exact headers and rates. The original Ottawa and UODS sensor formats are also supported as MAT files. With **Automatic** selected, CORA inputs return **Ambiguous Equipment**. An event-camera image resolves automatically only when its exact bytes occur in one of the two trusted local export indexes and both indexes are available. An unmatched image returns **Unknown Equipment**. A Drill Rig render is recognized only if it matches the exact bytes of an indexed render under `Images/drill_rig/`. Dimensions alone cannot establish equipment identity.

Equipment names here describe **laboratory component mappings**. **Whole-machine photo** mode uses an optional local detector for excavators, wheel loaders and dump trucks. It shows unverified visual suggestions. A photo of a mining machine does not establish the condition of its internal parts. The Drill Rig images are CAD renders. The separate Motor Grader gear-tooth checkpoint is built for small-gear tooth damage, not slewing-bearing faults.

In [1]:
from pathlib import Path
from functools import lru_cache
import sys
from tempfile import TemporaryDirectory

import ipywidgets as widgets
from IPython.display import display

cwd = Path.cwd()
notebook_dir = next(
    (base / "Notebook" for base in (cwd, *cwd.parents)
     if (base / "Notebook" / "pdm.py").is_file()),
    None,
)
if notebook_dir is None:
    raise FileNotFoundError("Open this notebook from the capstone repository.")
sys.path.insert(0, str(notebook_dir))
import pdm

def checkpoint(machine, modality):
    suffix = {"Sensor": "sensor.joblib", "Image": "images.pt"}[modality]
    return pdm.MODELS / f"{machine}_{suffix}"

def checkpoint_status(machine, modality):
    if machine == "grader" and modality == "Image":
        model = pdm.MODELS / "grader_gear_tooth.joblib"
        if model.is_file():
            return "Found (gear-tooth task)"
        if not pdm.TOOTH_ZIP.is_file():
            return "Not built — verified training export required"
        return "Not built — run grader train-final-images"
    return "Found" if checkpoint(machine, modality).is_file() else "Missing"

availability = [
    {
        "Equipment": "Motor Grader" if machine == "grader" else config.title,
        "Sensor checkpoint": checkpoint_status(machine, "Sensor"),
        "Image checkpoint": checkpoint_status(machine, "Image"),
    }
    for machine, config in pdm.MACHINES.items()
]
display(pdm.pd.DataFrame(availability).style.hide(axis="index"))

Equipment,Sensor checkpoint,Image checkpoint
Pump,Found,Found
Haul Truck,Found,Found
Belt Conveyor,Found,Found
Excavator,Found,Found
Dozer,Found,Found
Wheel Loader,Found,Found
Drill Rig,Found,Found
Motor Grader,Found,Not built — run grader train-final-images


### Upload and predict

Use trusted checkpoints from `Processed_Data/models/`. The notebook uses the saved sensor feature list and estimator, or the saved ResNet18 weights and the matching preprocessing in `pdm.py`. The repository’s final sensor training can select several algorithms, so the output reports the estimator actually saved. Confidence is the model’s class probability and has not been calibrated for field use. A checkpoint listed as found is validated when loaded.

The optional deployment commands are `python Notebook/pdm.py <equipment_key> train-final` for sensors and `python Notebook/pdm.py <equipment_key> train-final-images` for images. They create local checkpoints, separate from the published held-out evaluations. The notebook never runs these commands. Shared formats remain ambiguous even if all candidate checkpoints are installed.

To build the separate Motor Grader gear-tooth checkpoint, first place the verified Roboflow v1 COCO export at `Raw_Data/roboflow_gear_photos/GearBox_adddingImg.v1i.coco.zip`, then run `python Notebook/pdm.py grader train-final-images`. The command checks the archive SHA-256 before training. This model predicts good vs. damaged teeth on small gears; it does not predict Grader slewing-bearing faults.

For a shared format, the optional mapping is a user-supplied provenance claim. The notebook shows this in the result and only accepts a candidate compatible with the detected format. CORA gear-wear runs are reused for Belt Conveyor and Dozer, and CORA drive runs overlap across Excavator and Wheel Loader. The sample alone cannot establish a mining machine's identity.

The [dataset manifest](../Dataset_Manifest/manifest.md) lists the training data for each mapping. Deployment fits can include the recordings used in the examples, so the sample predictions only verify that inference runs. Refer to the existing held-out results for performance estimates.

Local deployment builds for Drill Rig and Motor Grader use the first two seconds of each publisher sensor recording. The Drill Rig image build uses a small subset of publisher CAD meshes. These checkpoints have a different training scope from the archived evaluations. The fault output describes current component condition. It provides no estimate of future failure or remaining life.

Choose **Whole-machine photo** for ordinary site photos. A single detection returns **Unknown Equipment** with an unverified suggestion and a box for visual review. Several detected categories return **Ambiguous Equipment**. The detector can confuse unsupported equipment with supported classes, including a Grader with a Wheel Loader. Its dump-truck label does not establish that the vehicle is a mining haul truck.

In [2]:
def identify_input(path):
    try:
        return pdm.identify(path)
    except ValueError as problem:
        return str(problem), [], {}


def score_checkpoint(path, machine, modality, source, details):
    weights = checkpoint(machine, modality)
    if modality == "Sensor":
        import joblib

        saved = joblib.load(weights)
        metadata = saved.get("metadata", {})
        if metadata and (metadata["machine"] != machine or metadata["modality"] != "sensor"):
            raise ValueError("Sensor checkpoint belongs to a different equipment or modality.")
        estimator = saved["model"]
        features = pdm.sensor_features(path, source, details)[saved["features"]]
        if features.empty:
            raise ValueError("The recording is too short for one complete feature window.")
        values = features.to_numpy(dtype=float)
        if not pdm.np.isfinite(values).all():
            raise ValueError("The recording produced invalid sensor features. Check the signal values.")
        scores = estimator.predict_proba(values).mean(axis=0)
        model_name = saved.get("model_name", type(estimator).__name__)
        return dict(zip(estimator.classes_, scores)), model_name

    return pdm.image_probability(path, machine), "ResNet18 CNN"


def predict_file(path, verified_mapping=None):
    extension = path.suffix.lower()
    if extension in {".csv", ".mat"}:
        modality = "Sensor"
    elif extension in {".png", ".jpg", ".jpeg"}:
        modality = "Image"
    else:
        return {"Status": "Unsupported file", "Accepted": "CSV, MAT, PNG, JPG, JPEG"}

    if verified_mapping == "grader" and modality == "Image":
        try:
            return pdm.predict_grader_tooth(path)
        except FileNotFoundError as problem:
            return {"Status": "Model unavailable", "Equipment": "Motor Grader",
                    "Task": "Gear-tooth visual inspection", "Reason": str(problem)}

    source, candidates, details = identify_input(path)
    result = {"Input": modality, "Source": source}
    if not candidates:
        return {**result, "Status": "Unknown Equipment"}
    if verified_mapping is not None and verified_mapping not in candidates:
        return {**result, "Status": "Mapping incompatible with detected format"}
    if len(candidates) > 1:
        names = ", ".join(pdm.MACHINES[machine].title for machine in candidates)
        if verified_mapping is None or details.get("mixed_schema"):
            return {**result, "Status": "Ambiguous Equipment", "Candidates": names}
        result["Identification"] = "User-confirmed laboratory mapping"
    else:
        result["Identification"] = (
            "Exact match in a trusted local image export" if details.get("indexed_image")
            else "Recognized dataset format"
        )

    machine = verified_mapping or candidates[0]
    result["Equipment"] = "Motor Grader" if machine == "grader" else pdm.MACHINES[machine].title
    if not checkpoint(machine, modality).is_file():
        return {**result, "Status": "Model unavailable", "Required checkpoint": checkpoint(machine, modality).name}
    probabilities, model_name = score_checkpoint(path, machine, modality, source, details)
    fault = max(probabilities, key=probabilities.get)
    return {**result, "Status": "Prediction", "Predicted fault class": fault,
            "Model confidence": f"{probabilities[fault]:.1%}", "Model used": model_name}


@lru_cache(maxsize=1)
def photo_detector():
    import json

    folder = pdm.MODELS / "whole_machine"
    config_file = folder / "config.json"
    if not config_file.is_file():
        raise FileNotFoundError("Install the optional local whole-machine detector described in README.md.")
    config = json.loads(config_file.read_text())
    import onnxruntime

    session = onnxruntime.InferenceSession(str(folder / config["onnx"]), providers=["CPUExecutionProvider"])
    return session, config


def identify_photo(path):
    import cv2
    import numpy as np

    image = cv2.imread(str(path))
    if image is None:
        return {"Status": "Unknown Equipment", "Reason": "Upload a valid image."}
    session, config = photo_detector()
    size = config["imgsz"]
    height, width = image.shape[:2]
    scale = min(size / width, size / height)
    new_width, new_height = max(1, round(width * scale)), max(1, round(height * scale))
    dx, dy = (size - new_width) // 2, (size - new_height) // 2
    canvas = np.full((size, size, 3), config["letterbox_pad"], dtype=np.uint8)
    rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    canvas[dy:dy + new_height, dx:dx + new_width] = cv2.resize(rgb, (new_width, new_height))
    tensor = np.ascontiguousarray(canvas.transpose(2, 0, 1)[None], dtype=np.float32) / 255
    detections = session.run(None, {session.get_inputs()[0].name: tensor})[0][0]
    scores = detections[4:]
    winners = scores.argmax(axis=0)
    candidates = {}
    for label in ("excavator", "wheel loader", "dump truck"):
        label_index = config["classes"].index(label)
        matches = np.flatnonzero((winners == label_index) & (scores[label_index] >= 0.60))
        for index in matches[np.argsort(scores[label_index, matches])[::-1]]:
            cx, cy, box_width, box_height = detections[:4, index]
            if not (dx <= cx < dx + new_width and dy <= cy < dy + new_height):
                continue
            left = max(0, round((cx - box_width / 2 - dx) / scale))
            top = max(0, round((cy - box_height / 2 - dy) / scale))
            right = min(width, round((cx + box_width / 2 - dx) / scale))
            bottom = min(height, round((cy + box_height / 2 - dy) / scale))
            if right <= left or bottom <= top:
                continue
            candidates[label.title()] = (float(scores[label_index, index]), (left, top, right, bottom))
            break

    if not candidates:
        return {"Status": "Unknown Equipment", "Reason": "No supported category cleared the 60% score threshold."}
    if len(candidates) > 1:
        names = ", ".join(f"{label} ({value[0]:.1%})" for label, value in candidates.items())
        return {"Status": "Ambiguous Equipment", "Candidates": names}
    equipment, (score, box) = next(iter(candidates.items()))
    return {"Status": "Unknown Equipment", "Unverified suggestion": equipment,
            "Detection score": f"{score:.1%}", "Candidate box (left, top, right, bottom)": box,
            "Model used": config["model"], "Fault prediction": "Unavailable for whole-machine photos"}

In [3]:
from html import escape

upload = widgets.FileUpload(
    accept=".csv,.mat,.png,.jpg,.jpeg", multiple=False,
    description="Choose file",
    style={"button_width": "150px"},
)
mode = widgets.ToggleButtons(
    options=["Component fault", "Whole-machine photo"],
    description="Task", style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
mapping = widgets.Dropdown(
    options=[("Automatic", None)] + [(row["Equipment"], key)
                                     for key, row in zip(pdm.MACHINES, availability)],
    description="Equipment mapping", style={"description_width": "initial"},
    layout=widgets.Layout(width="100%", max_width="420px"),
)
predict_button = widgets.Button(
    description="Run prediction", icon="search", button_style="primary",
    layout=widgets.Layout(width="180px", height="40px"),
)
output = widgets.Output()
task_help = widgets.HTML()

def update_task_help(change=None):
    is_photo = mode.value == "Whole-machine photo"
    mapping.disabled = is_photo
    task_help.value = (
        "<span>Upload a whole-machine image (PNG or JPG) for equipment identification.</span>"
        if is_photo else
        "<span>Upload sensor data (.csv or .mat) or a supported fault-data image.</span>"
    )

mode.observe(update_task_help, names="value")
update_task_help()

def show_message(message, kind="info"):
    colors = {"info": ("#eff6ff", "#1d4ed8"), "error": ("#fef2f2", "#b91c1c")}
    background, foreground = colors[kind]
    with output:
        display(widgets.HTML(
            value=f'<div style="padding:12px 14px;border-radius:8px;'
                  f'background:{background};color:{foreground};">{escape(message)}</div>'
        ))

def on_predict(_):
    output.clear_output(wait=True)
    if not upload.value:
        show_message("Choose a file before running a prediction.", "error")
        return
    uploaded = upload.value[0]
    name = Path(uploaded["name"]).name
    if name in {"", ".", ".."}:
        show_message("The selected filename is invalid.", "error")
        return
    predict_button.disabled = True
    predict_button.description = "Analyzing…"
    try:
        with TemporaryDirectory() as directory:
            path = Path(directory) / name
            path.write_bytes(uploaded["content"])
            if mode.value == "Whole-machine photo":
                result = identify_photo(path)
            else:
                result = predict_file(path, verified_mapping=mapping.value)
        rows = "".join(
            f"<tr><th>{escape(str(label))}</th><td>{escape(str(value))}</td></tr>"
            for label, value in result.items()
        )
        with output:
            display(widgets.HTML(
                value="<div style='border:1px solid #dbe3ee;border-radius:10px;"
                      "overflow:hidden;margin-top:8px'>"
                      "<div style='padding:12px 16px;background:#f8fafc;"
                      "font-weight:600;color:#0f172a'>Prediction results</div>"
                      f"<table style='width:100%;border-collapse:collapse'>{rows}</table>"
                      "</div><style>td,th{padding:10px 14px;text-align:left;"
                      "border-top:1px solid #e2e8f0}th{width:35%;color:#475569;"
                      "font-weight:500}td{color:#0f172a}</style>"
            ))
    except Exception as problem:
        show_message(f"Prediction unavailable: {problem}", "error")
    finally:
        predict_button.disabled = False
        predict_button.description = "Run prediction"

predict_button.on_click(on_predict)
header = widgets.HTML(
    "<div style='padding:4px 0 10px'>"
    "<div style='font-size:20px;font-weight:700;color:#0f172a'>Equipment diagnostics</div>"
    "<div style='color:#64748b;margin-top:4px'>Select an analysis task and provide a supported file.</div>"
    "</div>"
)
upload_row = widgets.HBox(
    [upload, predict_button], layout=widgets.Layout(gap="12px", align_items="center", flex_wrap="wrap")
)
display(widgets.VBox(
    [header, mode, task_help, mapping, upload_row, output],
    layout=widgets.Layout(border="1px solid #dbe3ee", border_radius="12px",
                          padding="20px", gap="12px", max_width="760px"),
))